In [1]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

class CUBDataset(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.root = root
        self.transform = transform
        self.train = train
        # Load image paths and labels
        images_file = os.path.join(root, 'images.txt')
        split_file = os.path.join(root, 'train_test_split.txt')
        labels_file = os.path.join(root, 'image_class_labels.txt')
        with open(images_file, 'r') as f:
            self.image_paths = [line.strip().split()[1] for line in f]
        with open(split_file, 'r') as f:
            self.is_train = [int(line.strip().split()[1]) for line in f]
        with open(labels_file, 'r') as f:
            self.labels = [int(line.strip().split()[1]) - 1 for line in f]  # 0-based
        # Filter for train or test
        if train:
            indices = [i for i, t in enumerate(self.is_train) if t == 1]
        else:
            indices = [i for i, t in enumerate(self.is_train) if t == 0]
        self.image_paths = [self.image_paths[i] for i in indices]
        self.labels = [self.labels[i] for i in indices]
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img_path = os.path.join(self.root, 'images', self.image_paths[idx])
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = CUBDataset("D:\Downloads\CUB_200_2011\CUB_200_2011", train=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Test one batch
images, labels = next(iter(train_loader))

print(images.shape)   # should be [8, 3, 224, 224]
print(labels.shape)   # should be [8]
print(labels[:5])

<>:49: SyntaxWarning: invalid escape sequence '\D'
<>:49: SyntaxWarning: invalid escape sequence '\D'
C:\Users\sitha\AppData\Local\Temp\ipykernel_22952\2152517734.py:49: SyntaxWarning: invalid escape sequence '\D'
  train_dataset = CUBDataset("D:\Downloads\CUB_200_2011\CUB_200_2011", train=True, transform=transform)


torch.Size([8, 3, 224, 224])
torch.Size([8])
tensor([ 37, 182,  87, 100,  36])


In [6]:
#check if GPU is available

import torch

divice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(divice)

cpu


In [7]:
from torchvision import models
model=models.resnet18(pretrained=True)

# load the pretrained model

In [3]:
for param in model.parameters():
    param.requires_grad = False
    
# freeze backbone 

In [4]:
# final layer

import torch.nn as nn
model.fc = nn.Linear(model.fc.in_features, 200)

In [8]:
model=model.to(divice)

In [9]:
# unfreeze deeper layers
for param in model.layer3.parameters():
    param.requires_grad = True
    
for param in model.layer4.parameters():
    param.requires_grad = True

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [11]:
images,labelsm = next(iter(train_loader))
images = images.to(divice)
labels = labels.to(divice)
outputs = model(images)
print(outputs.shape)  

torch.Size([8, 1000])


In [12]:
test_dataset = CUBDataset("D:/Downloads/CUB_200_2011/CUB_200_2011", train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [13]:
def top5_accuracy(outputs, labels):
    _, top5 = outputs.topk(5, dim=1)
    correct = top5.eq(labels.view(-1, 1).expand_as(top5))
    return correct.sum().item() 

In [18]:
import torch
from tqdm import tqdm

# ✅ define ONCE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = model.to(device)

num_epochs = 3

for epoch in range(num_epochs):

    # 🔥 TRAINING
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):
        images = images.to(device)   # ✅ FIX
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()*images.size(0)
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss= train_loss / total
    train_acc = 100 * correct / total

    # 🔵 VALIDATION
    model.eval()
    val_loss = 0
    correct = 0
    top5_correct=0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()*images.size(0)
            # top1
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            # top5
            _, top5 = outputs.topk(5, dim=1)
            top5_correct += top5.eq(labels.view(-1, 1)).sum().item()

            total += labels.size(0)
           

    val_loss = val_loss / total 
    val_acc = 100 * correct / total
    top5_acc = 100 * top5_correct / total

    print(f"\nEpoch {epoch+1}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Acc (top-1) strict accuracy:{val_acc:.2f}%")
    print(f"Val Acc (top-5) model understanding:{top5_acc:.2f}%")


Using: cpu


100%|██████████| 750/750 [11:16<00:00,  1.11it/s]



Epoch 1
Val Loss: 2.5380
Val Acc (top-1) strict accuracy:36.74%
Val Acc (top-5) model understanding:68.23%


100%|██████████| 750/750 [10:48<00:00,  1.16it/s]



Epoch 2
Val Loss: 2.0092
Val Acc (top-1) strict accuracy:48.88%
Val Acc (top-5) model understanding:78.63%


100%|██████████| 750/750 [10:27<00:00,  1.19it/s]



Epoch 3
Val Loss: 1.8667
Val Acc (top-1) strict accuracy:53.69%
Val Acc (top-5) model understanding:80.89%


In [20]:
torch.save(model.state_dict(), "cub_resnet18.pth")
print("Model saved as cub_resnet18.pth")

Model saved as cub_resnet18.pth
